# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Ranked actions and reason codes

This playbook converts the validated Logistic Regression score into a ranked
decision-support queue. A higher score means the page should be reviewed
earlier; it does not mean that a content change should happen automatically.

The actions are:

- **PRIORITY_REVIEW** — high model score and enough observed search visibility.
- **REVIEW** — elevated model score, but the evidence is less strong.
- **MONITOR** — lower score; no immediate content action is recommended.

Reason codes explain why a page entered the queue:

- **HIGH_DECLINE_SCORE_HIGH_VIS** — high decline score with substantial impressions.
- **HIGH_DECLINE_SCORE** — high decline score without the additional visibility signal.
- **MODERATE_DECLINE_SCORE** — moderate model score; monitor/review cautiously.

The queue is intended to prioritise human investigation. Before changing
content, a reviewer should consider ranking position, search intent, traffic
volume, freshness, and other context not captured by the model.

In [7]:
# SECTION 1 — RANKED ACTIONS + REASON CODES

from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# -----------------------------
# Load March 2026 data
# -----------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
print("Connected as:", api.whoami()["name"])

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(file_path)

available = march[
    march["gsc_data_available"].eq(True) &
    march["ga4_data_available"].eq(True)
].copy()

available["report_date"] = pd.to_datetime(
    available["report_date"]
)


# -----------------------------
# Same W05/W06 feature design
# -----------------------------

early = available[
    available["report_date"].between(
        "2026-03-01", "2026-03-15"
    )
].copy()

late = available[
    available["report_date"].between(
        "2026-03-16", "2026-03-31"
    )
].copy()


early_page = (
    early.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        early_days=("report_date", "nunique")
    )
)


late_page = (
    late.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        late_clicks=("gsc_clicks", "sum"),
        late_days=("report_date", "nunique")
    )
)


model_df = early_page.merge(
    late_page,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["early_avg_clicks"] = (
    model_df["clicks"] / model_df["early_days"]
)

model_df["late_avg_clicks"] = (
    model_df["late_clicks"] / model_df["late_days"]
)

model_df["declining"] = (
    model_df["late_avg_clicks"] <
    model_df["early_avg_clicks"]
).astype(int)


features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "engaged_sessions"
]

model_df = model_df.replace(
    [np.inf, -np.inf], np.nan
)

model_df = model_df.dropna(
    subset=features
).copy()


# -----------------------------
# Honest client-grouped split
# -----------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["declining"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()


model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(
    train_df[features],
    train_df["declining"]
)

queue = test_df.copy()

queue["model_score"] = model.predict_proba(
    queue[features]
)[:, 1]


# -----------------------------
# Action thresholds
# -----------------------------

visibility_threshold = train_df["impressions"].median()


def assign_action(row):

    if (
        row["model_score"] >= 0.70
        and row["impressions"] >= visibility_threshold
    ):
        return "PRIORITY_REVIEW"

    elif row["model_score"] >= 0.50:
        return "REVIEW"

    else:
        return "MONITOR"


def assign_reason(row):

    if (
        row["model_score"] >= 0.70
        and row["impressions"] >= visibility_threshold
    ):
        return "HIGH_DECLINE_SCORE_HIGH_VIS"

    elif row["model_score"] >= 0.70:
        return "HIGH_DECLINE_SCORE"

    else:
        return "MODERATE_DECLINE_SCORE"


queue["action"] = queue.apply(
    assign_action,
    axis=1
)

queue["reason_code"] = queue.apply(
    assign_reason,
    axis=1
)


# Rank highest-risk pages first
queue = queue.sort_values(
    ["model_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)


# -----------------------------
# Results
# -----------------------------

print("Model rows:", len(model_df))
print("Train rows:", len(train_df))
print("Queue rows:", len(queue))

print(
    "Client overlap:",
    len(
        set(train_df["client_hash_id"]) &
        set(test_df["client_hash_id"])
    )
)

print(
    "Visibility threshold:",
    visibility_threshold
)

print("\nACTION COUNTS")
print(queue["action"].value_counts())

print("\nREASON CODE COUNTS")
print(queue["reason_code"].value_counts())


display_cols = [
    "rank",
    "content_hash_id",
    "model_score",
    "impressions",
    "clicks",
    "avg_position",
    "action",
    "reason_code"
]

print("\nTOP 20 RANKED ACTIONS")
display(
    queue[display_cols].head(20)
)

Connected as: Krisha1903
Model rows: 29394
Train rows: 28227
Queue rows: 1167
Client overlap: 0
Visibility threshold: 175.0

ACTION COUNTS
action
MONITOR            961
REVIEW             190
PRIORITY_REVIEW     16
Name: count, dtype: int64

REASON CODE COUNTS
reason_code
MODERATE_DECLINE_SCORE         1151
HIGH_DECLINE_SCORE_HIGH_VIS      16
Name: count, dtype: int64

TOP 20 RANKED ACTIONS


,rank,content_hash_id,model_score,impressions,clicks,avg_position,action,reason_code
0,1,content_b5bef91d1b43e20c,0.999970,8095,172,9.022349,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
1,2,content_f2e852c493baaa19,0.991618,2663,84,9.745454,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
2,3,content_6098011c8d0d4c01,0.959427,2201,61,2.722972,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
3,4,content_a47fd3545d1a68c6,0.914311,2678,45,8.903121,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
4,5,content_b2347ac65344f8c0,0.881692,2068,37,1.793749,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
5,6,content_30352f0b93c8b98d,0.854179,1600,34,17.999995,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
6,7,content_24a12e5e6a9fa69a,0.839932,3392,31,4.980071,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
7,8,content_7b50820e76b1a006,0.831766,886,36,29.122832,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
8,9,content_0cfa3d9d68583bf1,0.814829,15712,25,5.395963,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
9,10,content_9a13341d5adaea3a,0.794375,1552,28,1.755968,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Intended use

This playbook is a decision-support tool for content and SEO teams. It ranks
pages so that limited review time can be focused on pages showing stronger
model-estimated decline risk.

The recommended workflow is:

1. Review **PRIORITY_REVIEW** pages first.
2. Check the reason code and supporting metrics.
3. Inspect the page's search intent, ranking position, traffic pattern and
   freshness before deciding on an action.
4. Use **REVIEW** pages as the second review tier.
5. Keep **MONITOR** pages under observation rather than changing them
   automatically.

The decay/refresh insight is directional: declining or stale-looking content
may be a candidate for refresh, but a refresh should only be recommended after
human review confirms that the content is outdated or no longer meets search
intent.

### Limits

The model was evaluated on March 2026 data and achieved a measured
Precision@50 of 0.70 on a client-grouped holdout with zero client overlap.

This result does not establish causality and does not guarantee the same
performance for other clients or future time periods. The model does not
directly understand content quality, search intent, business importance,
seasonality, SERP changes or the cost of editing a page.

Therefore, model scores should prioritise investigation, not automatically
trigger content changes.

In [9]:
# SECTION 2 — INTENDED USE AND LIMITS

playbook_summary = pd.DataFrame({
    "Action": ["PRIORITY_REVIEW", "REVIEW", "MONITOR"],
    "Human response": [
        "Review first; inspect context before any change",
        "Review after priority items",
        "Observe; no immediate content change"
    ],
    "Automation allowed": ["No", "No", "No"]
})

display(playbook_summary)

print("Validated Precision@50: 0.70")
print("Client overlap in validation: 0")
print("Scope: decision support, not automatic content editing")

,Action,Human response,Automation allowed
0,PRIORITY_REVIEW,Review first; inspect context before any change,No
1,REVIEW,Review after priority items,No
2,MONITOR,Observe; no immediate content change,No


Validated Precision@50: 0.70
Client overlap in validation: 0
Scope: decision support, not automatic content editing


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Human review rules

Every recommended page must be reviewed by a person before any content change.

The reviewer should check:

- whether the traffic decline is meaningful and not normal variation;
- current ranking position and search intent;
- whether the content is actually outdated or inaccurate;
- seasonality or temporary demand changes;
- business importance and expected value of improving the page;
- whether the likely benefit justifies the cost of editing.

High model score means higher review priority, not permission to edit.

### Cost/value thinking

Review effort should be concentrated on high-confidence, high-visibility pages
where an improvement could have greater value. Low-traffic or uncertain cases
should generally be monitored before spending significant editing effort.

### No-go list

The following should NOT be automated:

- rewriting, deleting or publishing content;
- changing titles, headings or metadata automatically;
- treating a high model score as proof that a page will decline;
- refreshing content only because the model assigned a high score;
- making decisions without checking search intent and business context;
- taking action on very low-evidence pages without human review.

The playbook ranks opportunities for investigation; the final action remains
a human decision.

In [11]:
# SECTION 3 — HUMAN REVIEW + NO-GO LIST

review_rules = pd.DataFrame({
    "Check": [
        "Traffic evidence",
        "Ranking and search intent",
        "Content freshness/accuracy",
        "Seasonality",
        "Business value",
        "Editing cost"
    ],
    "Required": ["Yes"] * 6
})

no_go = [
    "Automatic content rewriting",
    "Automatic deletion or publishing",
    "Automatic metadata changes",
    "Treating model score as proof of decline",
    "Refreshing solely because of model score",
    "Acting without human context review"
]

print("HUMAN REVIEW CHECKLIST")
display(review_rules)

print("\nNO-GO / DO NOT AUTOMATE")
for item in no_go:
    print("-", item)

print("\nFinal decision owner: Human reviewer")

HUMAN REVIEW CHECKLIST


,Check,Required
0,Traffic evidence,Yes
1,Ranking and search intent,Yes
2,Content freshness/accuracy,Yes
3,Seasonality,Yes
4,Business value,Yes
5,Editing cost,Yes



NO-GO / DO NOT AUTOMATE
- Automatic content rewriting
- Automatic deletion or publishing
- Automatic metadata changes
- Treating model score as proof of decline
- Refreshing solely because of model score
- Acting without human context review

Final decision owner: Human reviewer


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Monitoring and retrain triggers

The playbook should be monitored because search behaviour, client mix and
content performance can change over time.

The model should be reviewed or retrained when:

- Precision@50 drops below 0.60 on a new labelled evaluation window.
- the feature distributions change substantially from the March 2026 reference;
- a new client or content group behaves very differently from the training data;
- the balance of declining vs non-declining pages changes substantially;
- repeated human review shows that priority recommendations are often incorrect;
- the underlying data definitions or collection process changes.

A regular monthly check is appropriate for this prototype. Retraining should
not happen automatically. A human should first investigate whether the change
comes from genuine model drift, seasonality, data quality or another external
factor.

In [13]:
# SECTION 4 — MONITORING / RETRAIN TRIGGERS

monitoring_plan = pd.DataFrame({
    "Signal": [
        "Precision@50",
        "Feature distribution",
        "New client behaviour",
        "Decline base rate",
        "Human-review quality",
        "Data pipeline/schema"
    ],
    "Trigger": [
        "Falls below 0.60",
        "Substantial shift from reference data",
        "Behaviour differs strongly from training clients",
        "Substantial change from reference rate",
        "Priority recommendations repeatedly look wrong",
        "Feature definition or collection changes"
    ],
    "Response": [
        "Investigate and consider retraining",
        "Check drift before using scores",
        "Validate separately before relying on scores",
        "Recheck thresholds and calibration",
        "Audit errors and model assumptions",
        "Pause scoring and validate pipeline"
    ]
})

print("MONITORING / RETRAIN PLAN")
display(monitoring_plan)

print("\nReference grouped Precision@50: 0.70")
print("Suggested monitoring frequency: monthly")
print("Retraining decision: human-reviewed, not automatic")

MONITORING / RETRAIN PLAN


,Signal,Trigger,Response
0,Precision@50,Falls below 0.60,Investigate and consider retraining
1,Feature distribution,Substantial shift from reference data,Check drift before using scores
2,New client behaviour,Behaviour differs strongly from training clients,Validate separately before relying on scores
3,Decline base rate,Substantial change from reference rate,Recheck thresholds and calibration
4,Human-review quality,Priority recommendations repeatedly look wrong,Audit errors and model assumptions
5,Data pipeline/schema,Feature definition or collection changes,Pause scoring and validate pipeline



Reference grouped Precision@50: 0.70
Suggested monitoring frequency: monthly
Retraining decision: human-reviewed, not automatic


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Exports for the paper

This notebook exports the ranked action queue and a small metrics receipt for
reuse in the capstone paper.

The ranked queue contains hashed content identifiers, model scores, actions,
reason codes and supporting aggregate metrics. It does not contain client
names, URLs or private search queries.

The CSV is generated from this notebook and is intentionally not committed
because repository leak protection excludes data files. The metrics JSON is
the reproducible receipt for the headline validation and playbook numbers used
in the paper.

In [15]:
# SECTION 5 — EXPORTS FOR THE PAPER

import os
import json

os.makedirs("work/outputs", exist_ok=True)

# -----------------------------
# Export ranked action queue
# -----------------------------

export_cols = [
    "rank",
    "content_hash_id",
    "model_score",
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "engaged_sessions",
    "action",
    "reason_code"
]

queue_export = queue[export_cols].copy()

queue_path = "work/outputs/content_action_playbook.csv"

queue_export.to_csv(
    queue_path,
    index=False
)


# -----------------------------
# Export metrics receipt
# -----------------------------

metrics = {
    "reference_period": "2026-03",
    "validation_design": "grouped_by_client",
    "client_overlap": 0,
    "precision_at_50": 0.70,
    "queue_rows": int(len(queue)),
    "priority_review_rows": int(
        (queue["action"] == "PRIORITY_REVIEW").sum()
    ),
    "review_rows": int(
        (queue["action"] == "REVIEW").sum()
    ),
    "monitor_rows": int(
        (queue["action"] == "MONITOR").sum()
    ),
    "visibility_threshold": float(visibility_threshold),
    "intended_use": "human-reviewed decision support"
}

metrics_path = "work/outputs/w07_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)


# -----------------------------
# Verify exports
# -----------------------------

print("EXPORT COMPLETE")
print("Queue:", queue_path)
print("Metrics:", metrics_path)

print("\nQueue rows:", len(queue_export))
print("Priority review:", metrics["priority_review_rows"])
print("Review:", metrics["review_rows"])
print("Monitor:", metrics["monitor_rows"])

print("\nMetrics receipt:")
print(json.dumps(metrics, indent=2))

print("\nQueue preview:")
display(queue_export.head(10))

EXPORT COMPLETE
Queue: work/outputs/content_action_playbook.csv
Metrics: work/outputs/w07_playbook_metrics.json

Queue rows: 1167
Priority review: 16
Review: 190
Monitor: 961

Metrics receipt:
{
  "reference_period": "2026-03",
  "validation_design": "grouped_by_client",
  "client_overlap": 0,
  "precision_at_50": 0.7,
  "queue_rows": 1167,
  "priority_review_rows": 16,
  "review_rows": 190,
  "monitor_rows": 961,
  "visibility_threshold": 175.0,
  "intended_use": "human-reviewed decision support"
}

Queue preview:


,rank,content_hash_id,model_score,impressions,clicks,avg_position,pageviews,engaged_sessions,action,reason_code
0,1,content_b5bef91d1b43e20c,0.999970,8095,172,9.022349,356.0,22.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
1,2,content_f2e852c493baaa19,0.991618,2663,84,9.745454,159.0,8.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
2,3,content_6098011c8d0d4c01,0.959427,2201,61,2.722972,32.0,1.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
3,4,content_a47fd3545d1a68c6,0.914311,2678,45,8.903121,102.0,2.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
4,5,content_b2347ac65344f8c0,0.881692,2068,37,1.793749,27.0,3.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
5,6,content_30352f0b93c8b98d,0.854179,1600,34,17.999995,62.0,4.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
6,7,content_24a12e5e6a9fa69a,0.839932,3392,31,4.980071,62.0,2.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
7,8,content_7b50820e76b1a006,0.831766,886,36,29.122832,68.0,1.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
8,9,content_0cfa3d9d68583bf1,0.814829,15712,25,5.395963,32.0,0.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS
9,10,content_9a13341d5adaea3a,0.794375,1552,28,1.755968,11.0,0.0,PRIORITY_REVIEW,HIGH_DECLINE_SCORE_HIGH_VIS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.